# Road Following - Live demo

In this notebook, we will use model we trained to move jetBot smoothly on track. 

### Load Trained Model

We will assume that you have already downloaded ``best_steering_model_xy.pth`` to work station as instructed in "train_model.ipynb" notebook. Now, you should upload model file to JetBot in to this notebook's directory. Once that's finished there should be a file named ``best_steering_model_xy.pth`` in this notebook's directory.

> Please make sure the file has uploaded fully before calling the next cell

Execute the code below to initialize the PyTorch model. This should look very familiar from the training notebook.

In [1]:
MODEL_PATH = 'model_to_import.pth'

In [2]:
import torchvision
import torch

model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Next, load the trained weights from the ``best_steering_model_xy.pth`` file that you uploaded.

In [3]:
model.load_state_dict(torch.load(MODEL_PATH))

<All keys matched successfully>

Currently, the model weights are located on the CPU memory execute the code below to transfer to the GPU device.

In [4]:
device = torch.device('cuda')
model = model.to(device)
model = model.eval().float()

### Creating the Pre-Processing Function

We have now loaded our model, but there's a slight issue. The format that we trained our model doesn't exactly match the format of the camera. To do that, we need to do some preprocessing. This involves the following steps:

1. Convert from HWC layout to CHW layout
2. Normalize using same parameters as we did during training (our camera provides values in [0, 255] range and training loaded images in [0, 1] range so we need to scale by 255.0
3. Transfer the data from CPU memory to GPU memory
4. Add a batch dimension

In [5]:
import torchvision.transforms as transforms
import torch.nn.functional as F
import cv2
import PIL.Image
import numpy as np

mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device)
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

Awesome! We've now defined our pre-processing function which can convert images from the camera format to the neural network input format.

Now, let's start and display our camera. You should be pretty familiar with this by now. 

In [6]:
from IPython.display import display
import ipywidgets
import traitlets
from jetbot import Camera, bgr8_to_jpeg

camera = Camera()

image_widget = ipywidgets.Image()

traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

We'll also create our robot instance which we'll need to drive the motors.

In [7]:
from jetbot import Robot

robot = Robot()

Now, we will define sliders to control JetBot
> Note: We have initialize the slider values for best known configurations, however these might not work for your dataset, therefore please increase or decrease the sliders according to your setup and environment

1. Speed Control (speed_gain_slider): To start your JetBot increase ``speed_gain_slider`` 
2. Steering Gain Control (steering_gain_slider): If you see JetBot is wobbling, you need to reduce ``steering_gain_slider`` till it is smooth
3. Steering Bias control (steering_bias_slider): If you see JetBot is biased towards extreme right or extreme left side of the track, you should control this slider till JetBot start following line or track in the center.  This accounts for motor biases as well as camera offsets

> Note: You should play around above mentioned sliders with lower speed to get smooth JetBot road following behavior.

In [8]:
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.2, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.0, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=0.0, description='steering bias')

display(speed_gain_slider, steering_gain_slider, steering_dgain_slider, steering_bias_slider)

FloatSlider(value=0.0, description='speed gain', max=1.0, step=0.01)

FloatSlider(value=0.2, description='steering gain', max=1.0, step=0.01)

FloatSlider(value=0.0, description='steering kd', max=0.5, step=0.001)

FloatSlider(value=0.0, description='steering bias', max=0.3, min=-0.3, step=0.01)

Next, let's display some sliders that will let us see what JetBot is thinking.  The x and y sliders will display the predicted x, y values.

The steering slider will display our estimated steering value.  Please remember, this value isn't the actual angle of the target, but simply a value that is
nearly proportional.  When the actual angle is ``0``, this will be zero, and it will increase / decrease with the actual angle.  

In [9]:
x_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='x')
y_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='y')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='speed')

display(ipywidgets.HBox([y_slider, speed_slider]))
display(x_slider, steering_slider)

FloatSlider(value=0.0, description='x', max=1.0, min=-1.0)

FloatSlider(value=0.0, description='steering', max=1.0, min=-1.0)

Next, we'll create a function that will get called whenever the camera's value changes. This function will do the following steps

1. Pre-process the camera image
2. Execute the neural network
3. Compute the approximate steering value
4. Control the motors using proportional / derivative control (PD)

In [18]:
robot.left_motor.value = 0
robot.right_motor.value = 0


In [12]:
import time

angle = 0.0
angle_last = 0.0

def execute(change):
    global angle, angle_last
    image = change['new']
    xy = model(preprocess(image)).detach().cuda().half().cpu().numpy().flatten()
    # x = xy[0]
    # y = (0.5 - xy[1]) / 2.0

    # CHANGED!
    x = xy[0] - 0.5   # re-center to [-0.5, 0.5]
    y = 1 - xy[1] # closer target = larger y → positive steering
    
    x_slider.value = x
    y_slider.value = y
    
    speed_slider.value = speed_gain_slider.value
    
    angle = np.arctan2(-x, y)
    pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
    angle_last = angle
    
    steering_slider.value = pid + steering_bias_slider.value
    
    print(x,y)
    print(max(min(speed_slider.value - steering_slider.value, 1.0), 0.0))
    print(max(min(speed_slider.value + steering_slider.value, 1.0), 0.0))
    robot.left_motor.value = max(min(speed_slider.value - steering_slider.value, 1.0), 0.0)
    robot.right_motor.value = max(min(speed_slider.value + steering_slider.value, 1.0), 0.0)
#     robot.stop()
    
# execute({'new': camera.value})

-0.0947265625 0.4951171875
0.24920591989173665
0.29079408010826335
-0.097412109375 0.4970703125
0.24871280256905048
0.29128719743094955
-0.095703125 0.49609375
0.2490370453810695
0.2909629546189305
-0.09521484375 0.49267578125
0.24900021885832005
0.29099978114168
-0.09130859375 0.49267578125
0.24984219508242264
0.29015780491757737
-0.09326171875 0.49169921875
0.2493809796597217
0.2906190203402783
-0.09326171875 0.49072265625
0.24934090397357017
0.29065909602642986
-0.0908203125 0.490234375
0.24984996360709627
0.29015003639290377
-0.0908203125 0.4921875
0.24992815756542247
0.29007184243457756
-0.08154296875 0.4951171875
0.25204481143839946
0.2879551885616006
-0.076904296875 0.486328125
0.25274827346438267
0.28725172653561737
-0.080078125 0.48486328125
0.25199535722676275
0.2880046427732373
-0.076904296875 0.4873046875
0.25278228358830457
0.28721771641169547
-0.07861328125 0.482421875
0.2522310817579365
0.2877689182420635
-0.07421875 0.48876953125
0.25342333668842
0.28657666331158005
-0.

Cool! We've created our neural network execution function, but now we need to attach it to the camera for processing.

We accomplish that with the observe function.

>WARNING: This code will move the robot!! Please make sure your robot has clearance and it is on Lego or Track you have collected data on. The road follower should work, but the neural network is only as good as the data it's trained on!

In [13]:
camera.observe(execute, names='value')

-0.057373046875 0.516357421875
0.2578277056708715
0.28217229432912855
-0.061279296875 0.514404296875
0.2569575250098425
0.28304247499015756
-0.061279296875 0.514404296875
0.2569575250098425
0.28304247499015756
-0.0625 0.52001953125
0.25684245455570587
0.28315754544429417
-0.0625 0.52001953125
0.25684245455570587
0.28315754544429417
-0.063720703125 0.51806640625
0.2565379254301359
0.2834620745698641
-0.063720703125 0.51806640625
0.2565379254301359
0.2834620745698641
-0.06005859375 0.5146484375
0.2572209889594594
0.28277901104054065
-0.06005859375 0.5146484375
0.2572209889594594
0.28277901104054065
-0.052978515625 0.509765625
0.25860890143837895
0.2813910985616211
-0.052978515625 0.509765625
0.25860890143837895
0.2813910985616211
-0.054443359375 0.51416015625
0.2583955683845081
0.2816044316154919
-0.054443359375 0.51416015625
0.2583955683845081
0.2816044316154919
-0.0537109375 0.510986328125
0.2584799517916578
0.28152004820834226
-0.0537109375 0.510986328125
0.2584799517916578
0.28152004

Awesome! If your robot is plugged in it should now be generating new commands with each new camera frame. 

You can now place JetBot on  Lego or Track you have collected data on and see whether it can follow track.

If you want to stop this behavior, you can unattach this callback by executing the code below.

In [16]:
import time

camera.unobserve(execute, names='value')

time.sleep(0.1)  # add a small sleep to make sure frames have finished processing

robot.stop()

0.24755859375 0.39208984375
0.3319493255713033
0.2080506744286967
0.23095703125 0.38671875
0.32922119646808534
0.2107788035319147


ValueError: list.remove(x): x not in list

0.23095703125 0.37939453125
0.33015126357689856
0.20984873642310148
0.2373046875 0.38720703125
0.33048059729258006
0.20951940270741995
0.23681640625 0.39404296875
0.32952629073507234
0.21047370926492767
0.23291015625 0.39794921875
0.3282475001386025
0.21175249986139752
0.22802734375 0.39794921875
0.32723680009041317
0.2127631999095869
0.234375 0.3974609375
0.32860768576930094
0.2113923142306991
0.2431640625 0.4072265625
0.32921309453100045
0.21078690546899956
0.2392578125 0.4111328125
0.32797440186933285
0.2120255981306672
0.23681640625 0.4052734375
0.32817055546650414
0.2118294445334959
0.22216796875 0.41015625
0.324606502874537
0.21539349712546302
0.22216796875 0.41015625
0.324606502874537
0.21539349712546302
0.20849609375 0.40625
0.32215853527323235
0.2178414647267677
0.22509765625 0.40283203125
0.3260521607294064
0.21394783927059363
0.19921875 0.4013671875
0.32067960047981137
0.21932039952018864
0.20263671875 0.400390625
0.32153651257041166
0.2184634874295884
0.205078125 0.40087890

Again, let's close the camera conneciton properly so that we can use the camera in other notebooks.

In [17]:
camera.stop()

0.21240234375 0.40966796875
0.3226149404329015
0.21738505956709853


### Conclusion
That's it for this live demo! Hopefully you had some fun seeing your JetBot moving smoothly on track following the road!!!

If your JetBot wasn't following road very well, try to spot where it fails. The beauty is that we can collect more data for these failure scenarios and the JetBot should get even better :)